# ATLAS - Logistics Risk Analysis

## Purpose

For this notebook, the goal is to combine the cleaned hospital, traffic, and synthetic shipment datasets.

The analysis will use historical traffic patterns and shipment information to identify which deliveries may require additional attention.

The final outputs from this notebook will be used to build the ATLAS dashboard.

## Step 1: Import Libraries

The first step is to import the Python libraries needed for the analysis.

Pandas will be used to load, combine, and analyze the processed datasets.

In [18]:
import pandas as pd

## Step 2: Load the Processed Datasets

The cleaned hospital and traffic datasets from the previous notebooks will be loaded along with the synthetic shipment dataset.

These three datasets will be combined during the analysis.

In [19]:
hospitals_df = pd.read_csv("../data/processed/nj_hospitals_clean.csv")
traffic_df = pd.read_csv("../data/processed/nj_traffic_clean.csv")
shipments_df = pd.read_csv("../data/processed/synthetic_shipments.csv")

print("Hospitals:", hospitals_df.shape)
print("Traffic incidents:", traffic_df.shape)
print("Shipments:", shipments_df.shape)

Hospitals: (78, 7)
Traffic incidents: (140718, 19)
Shipments: (50, 11)


## Step 3: Analyze Historical Traffic Risk by County

The traffic dataset will be grouped by county to understand where incidents have historically occurred most often.

For each county, the analysis will calculate:

- Total number of traffic incidents
- Average incident severity
- Average valid incident duration

These values will later help ATLAS estimate transportation risk for shipments traveling to hospitals in different parts of New Jersey.

In [20]:
valid_duration_df = traffic_df[
    traffic_df["Valid_Duration"] == True
].copy()

county_risk = traffic_df.groupby("County").agg(
    Incident_Count=("ID", "count"),
    Average_Severity=("Severity", "mean")
).reset_index()

duration_summary = valid_duration_df.groupby("County").agg(
    Average_Duration_Min=("Incident_Duration_Min", "mean")
).reset_index()

county_risk = county_risk.merge(
    duration_summary,
    on="County",
    how="left"
)

county_risk.head()

,County,Incident_Count,Average_Severity,Average_Duration_Min
0,Atlantic,2680,2.204851,123.711973
1,Bergen,18578,2.224621,129.571182
2,Burlington,7695,2.267706,115.159543
3,Camden,11992,2.256421,110.263280
4,Cape May,957,2.237200,121.181731


In [21]:
county_risk.sort_values(
    "Incident_Count",
    ascending=False
).head(10)

,County,Incident_Count,Average_Severity,Average_Duration_Min
1,Bergen,18578,2.224621,129.571182
6,Essex,16997,2.184562,117.632734
11,Middlesex,15527,2.228570,108.274004
19,Union,12214,2.201818,113.806117
3,Camden,11992,2.256421,110.263280
8,Hudson,11624,2.190640,124.551935
2,Burlington,7695,2.267706,115.159543
10,Mercer,6194,2.276881,112.342601
12,Monmouth,6132,2.184768,119.319265
15,Passaic,6104,2.184961,109.002624


## Step 4: Create a County Traffic Risk Score

The county-level traffic metrics use different scales, so they will first be normalized to values between 0 and 1.

ATLAS will then combine the metrics into a simple historical traffic-risk score using:

- 50% Incident Count
- 30% Average Severity
- 20% Average Incident Duration

A higher score represents a county with historically greater traffic-related risk.

In [22]:
county_risk["Incident_Count_Norm"] = (
    county_risk["Incident_Count"] / county_risk["Incident_Count"].max()
)

county_risk["Severity_Norm"] = (
    county_risk["Average_Severity"] / county_risk["Average_Severity"].max()
)

county_risk["Duration_Norm"] = (
    county_risk["Average_Duration_Min"] / county_risk["Average_Duration_Min"].max()
)

county_risk["Traffic_Risk_Score"] = (
    county_risk["Incident_Count_Norm"] * 0.50
    + county_risk["Severity_Norm"] * 0.30
    + county_risk["Duration_Norm"] * 0.20
)

county_risk.sort_values(
    "Traffic_Risk_Score",
    ascending=False
)[
    [
        "County",
        "Incident_Count",
        "Average_Severity",
        "Average_Duration_Min",
        "Traffic_Risk_Score"
    ]
].head(10)

,County,Incident_Count,Average_Severity,Average_Duration_Min,Traffic_Risk_Score
1,Bergen,18578,2.224621,129.571182,0.927699
6,Essex,16997,2.184562,117.632734,0.864641
11,Middlesex,15527,2.228570,108.274004,0.817645
19,Union,12214,2.201818,113.806117,0.732790
8,Hudson,11624,2.190640,124.551935,0.729956
3,Camden,11992,2.256421,110.263280,0.728350
2,Burlington,7695,2.267706,115.159543,0.620523
10,Mercer,6194,2.276881,112.342601,0.577422
12,Monmouth,6132,2.184768,119.319265,0.574498
13,Morris,6002,2.335721,107.066736,0.571964


## Step 5: Standardize County Names

The traffic dataset and shipment dataset use different capitalization for county names.

For example, the traffic data may list `Mercer` while the shipment data lists `MERCER`.

Because pandas requires exact text matches when merging datasets, both county columns will be converted to uppercase before the datasets are combined.

In [23]:
county_risk["County"] = county_risk["County"].str.upper()
shipments_df["Destination_County"] = shipments_df["Destination_County"].str.upper()

## Step 6: Add Traffic Risk to Each Shipment

Each shipment already has a destination county.

The county traffic-risk score will now be added to each shipment so ATLAS can compare shipment timing risk with historical transportation risk.

In [24]:
shipments_df = shipments_df.merge(
    county_risk[["County", "Traffic_Risk_Score"]],
    left_on="Destination_County",
    right_on="County",
    how="left"
)

shipments_df = shipments_df.drop(columns=["County"])

shipments_df[
    [
        "Shipment_ID",
        "Destination_Hospital",
        "Destination_County",
        "Shipment_Status",
        "Traffic_Risk_Score"
    ]
].head(10)

,Shipment_ID,Destination_Hospital,Destination_County,Shipment_Status,Traffic_Risk_Score
0,AT-1001,CAPITAL HEALTH REGIONAL MEDICAL CENTER,MERCER,Low Risk,0.577422
1,AT-1002,ST JOSEPH'S UNIVERSITY MEDICAL CENTER INC,PASSAIC,High Risk,0.560012
2,AT-1003,NORTHBROOK BEHAVIORAL HEALTH HOSPITAL,CAMDEN,Medium Risk,0.728350
3,AT-1004,THE UNIVERSITY HOSPITAL,ESSEX,Medium Risk,0.864641
4,AT-1005,NEWTON MEDICAL CENTER,SUSSEX,High Risk,0.514641
5,AT-1006,SAINT BARNABAS BEHAVIORAL HEALTH CENTER,OCEAN,Low Risk,0.546398
6,AT-1007,SAINT BARNABAS BEHAVIORAL HEALTH CENTER,OCEAN,Low Risk,0.546398
7,AT-1008,INSPIRA MEDICAL CENTER VINELAND,CUMBERLAND,Low Risk,0.491535
8,AT-1009,PALISADES MEDICAL CENTER,HUDSON,High Risk,0.729956
9,AT-1010,VIRTUA OUR LADY OF LOURDES HOSPITAL,CAMDEN,Low Risk,0.728350


## Step 7: Create an Overall ATLAS Risk Score

Each shipment already has a shipment risk level based on its remaining delivery buffer.

The county traffic-risk score will now be combined with the shipment timing risk to create one overall ATLAS risk score.

For this proof of concept:

- Shipment timing risk will make up 60% of the score
- Historical traffic risk will make up 40% of the score

A higher score means the shipment may require more attention.

In [26]:
risk_mapping = {
    "Low Risk": 0.33,
    "Medium Risk": 0.66,
    "High Risk": 1.00
}

shipments_df["Timing_Risk_Score"] = shipments_df[
    "Shipment_Status"
].map(risk_mapping)

shipments_df["ATLAS_Risk_Score"] = (
    shipments_df["Timing_Risk_Score"] * 0.60
    + shipments_df["Traffic_Risk_Score"] * 0.40
)

shipments_df[
    [
        "Shipment_ID",
        "Shipment_Status",
        "Timing_Risk_Score",
        "Traffic_Risk_Score",
        "ATLAS_Risk_Score"
    ]
].head(10)

,Shipment_ID,Shipment_Status,Timing_Risk_Score,Traffic_Risk_Score,ATLAS_Risk_Score
0,AT-1001,Low Risk,0.33,0.577422,0.428969
1,AT-1002,High Risk,1.00,0.560012,0.824005
2,AT-1003,Medium Risk,0.66,0.728350,0.687340
3,AT-1004,Medium Risk,0.66,0.864641,0.741856
4,AT-1005,High Risk,1.00,0.514641,0.805856
5,AT-1006,Low Risk,0.33,0.546398,0.416559
6,AT-1007,Low Risk,0.33,0.546398,0.416559
7,AT-1008,Low Risk,0.33,0.491535,0.394614
8,AT-1009,High Risk,1.00,0.729956,0.891982
9,AT-1010,Low Risk,0.33,0.728350,0.489340


## Step 8: Assign Final ATLAS Risk Levels

The overall ATLAS risk score will be converted into a simple final risk category.

For this proof of concept:

- Low Risk: score below 0.50
- Medium Risk: score from 0.50 to 0.75
- High Risk: score above 0.75

This makes the final dashboard easier to read and helps prioritize shipments that may require attention.

In [28]:
def assign_atlas_risk(score):
    if score < 0.50:
        return "Low Risk"
    elif score <= 0.75:
        return "Medium Risk"
    else:
        return "High Risk"

shipments_df["ATLAS_Risk_Level"] = shipments_df[
    "ATLAS_Risk_Score"
].apply(assign_atlas_risk)

shipments_df[
    [
        "Shipment_ID",
        "Shipment_Status",
        "Traffic_Risk_Score",
        "ATLAS_Risk_Score",
        "ATLAS_Risk_Level"
    ]
].head(10)

,Shipment_ID,Shipment_Status,Traffic_Risk_Score,ATLAS_Risk_Score,ATLAS_Risk_Level
0,AT-1001,Low Risk,0.577422,0.428969,Low Risk
1,AT-1002,High Risk,0.560012,0.824005,High Risk
2,AT-1003,Medium Risk,0.728350,0.687340,Medium Risk
3,AT-1004,Medium Risk,0.864641,0.741856,Medium Risk
4,AT-1005,High Risk,0.514641,0.805856,High Risk
5,AT-1006,Low Risk,0.546398,0.416559,Low Risk
6,AT-1007,Low Risk,0.546398,0.416559,Low Risk
7,AT-1008,Low Risk,0.491535,0.394614,Low Risk
8,AT-1009,High Risk,0.729956,0.891982,High Risk
9,AT-1010,Low Risk,0.728350,0.489340,Low Risk


## Step 9: Create the Dashboard-Ready Shipment Table

The final shipment dataset will be sorted by ATLAS risk score so the highest-risk shipments appear first.

This will make it easier for the dashboard to highlight deliveries that may require the most attention.

In [29]:
dashboard_shipments = shipments_df.sort_values(
    "ATLAS_Risk_Score",
    ascending=False
).reset_index(drop=True)

dashboard_shipments.head(10)

,Shipment_ID,Origin,Destination_Hospital,Destination_City,Destination_County,Scheduled_Arrival,Estimated_Arrival,Delivery_Deadline,Remaining_Buffer_Min,Shipment_Status,Temperature_C,Traffic_Risk_Score,Timing_Risk_Score,ATLAS_Risk_Score,ATLAS_Risk_Level
0,AT-1047,"Raritan, NJ",NEWARK BETH ISRAEL MEDICAL CENTER,NEWARK,ESSEX,2026-10-23 07:00:00,2026-10-23 07:57:00,2026-10-23 08:00:00,3.0,High Risk,3.0,0.864641,1.0,0.945856,High Risk
1,AT-1037,"Raritan, NJ",CLARA MAASS MEDICAL CENTER,BELLEVILLE,ESSEX,2026-10-23 02:00:00,2026-10-23 02:47:00,2026-10-23 03:00:00,13.0,High Risk,7.3,0.864641,1.0,0.945856,High Risk
2,AT-1011,"Raritan, NJ",SAINT MICHAEL'S MEDICAL CENTER,NEWARK,ESSEX,2026-10-22 13:00:00,2026-10-22 13:38:00,2026-10-22 14:00:00,22.0,High Risk,6.3,0.864641,1.0,0.945856,High Risk
3,AT-1012,"Raritan, NJ",NEWARK BETH ISRAEL MEDICAL CENTER,NEWARK,ESSEX,2026-10-22 13:30:00,2026-10-22 14:17:00,2026-10-22 14:30:00,13.0,High Risk,3.4,0.864641,1.0,0.945856,High Risk
4,AT-1016,"Raritan, NJ",CHILDREN'S SPECIALIZED HOSPITAL,NEW BRUNSWICK,MIDDLESEX,2026-10-22 15:30:00,2026-10-22 16:28:00,2026-10-22 16:30:00,2.0,High Risk,2.7,0.817645,1.0,0.927058,High Risk
5,AT-1048,"Raritan, NJ",HUDSON REGIONAL HOSPITAL,SECAUCUS,HUDSON,2026-10-23 07:30:00,2026-10-23 08:26:00,2026-10-23 08:30:00,4.0,High Risk,5.2,0.729956,1.0,0.891982,High Risk
6,AT-1017,"Raritan, NJ",HUDSON REGIONAL HOSPITAL,SECAUCUS,HUDSON,2026-10-22 16:00:00,2026-10-22 16:46:00,2026-10-22 17:00:00,14.0,High Risk,4.6,0.729956,1.0,0.891982,High Risk
7,AT-1009,"Raritan, NJ",PALISADES MEDICAL CENTER,NORTH BERGEN,HUDSON,2026-10-22 12:00:00,2026-10-22 12:38:00,2026-10-22 13:00:00,22.0,High Risk,7.8,0.729956,1.0,0.891982,High Risk
8,AT-1032,"Raritan, NJ",JEFFERSON STRATFORD HOSPITAL,STRATFORD,CAMDEN,2026-10-22 23:30:00,2026-10-23 00:19:00,2026-10-23 00:30:00,11.0,High Risk,3.8,0.728350,1.0,0.891340,High Risk
9,AT-1022,"Raritan, NJ",VIRTUA OUR LADY OF LOURDES HOSPITAL,CAMDEN,CAMDEN,2026-10-22 18:30:00,2026-10-22 19:16:00,2026-10-22 19:30:00,14.0,High Risk,6.2,0.728350,1.0,0.891340,High Risk


## Step 10: Save the Final Analysis Dataset

The completed shipment analysis will now be saved in the `data/processed` folder.

This file will be used as the main data source for the ATLAS dashboard.

In [31]:
dashboard_shipments.to_csv(
    "../data/processed/atlas_dashboard_data.csv",
    index=False
)